# Fase 4 — Transformación y Feature Engineering
**Input:** `data/clean/pizza_data_clean.csv`  
**Output:** `data/clean/pizza_features.csv`

Columnas calculadas que necesita el análisis de Fase 5:
- Temporales: `month`, `month_name`, `day_of_week`, `day_name_es`, `is_weekend`, `quarter`, `week`, `hour`, `time_block`
- Métricas: `revenue` (ya existe), `ticket_total` por pedido
- Dimensiones: `size_label` (legible), `size_order` (para ordenar en PBI)

In [ ]:
import pandas as pd
import numpy as np

CLEAN = '../data/clean/'
df = pd.read_csv(CLEAN + 'pizza_data_clean.csv', parse_dates=['date'])
print('Shape original:', df.shape)

## 4.1 — Columnas temporales

In [ ]:
df['month']       = df['date'].dt.month
df['month_name']  = df['date'].dt.strftime('%b')
df['day_of_week'] = df['date'].dt.dayofweek          # 0=Lunes, 6=Domingo
df['day_name']    = df['date'].dt.strftime('%A')
df['day_name_es'] = df['date'].dt.dayofweek.map({
    0:'Lunes', 1:'Martes', 2:'Miércoles', 3:'Jueves',
    4:'Viernes', 5:'Sábado', 6:'Domingo'
})
df['is_weekend']  = df['day_of_week'].isin([5, 6]).astype(int)
df['quarter']     = df['date'].dt.quarter
df['week']        = df['date'].dt.isocalendar().week.astype(int)

# Hora desde la columna time (viene como string en el CSV)
df['hour'] = pd.to_datetime(df['time'], format='%H:%M:%S').dt.hour

df['time_block'] = pd.cut(df['hour'],
    bins=[0, 11, 14, 17, 20, 24],
    labels=['Mañana (< 12)', 'Almuerzo (12-14)', 'Tarde (15-17)', 'Noche (18-20)', 'Cierre (> 20)'],
    right=False
)

# Orden cronológico para Power BI
time_block_order_map = {
    'Mañana (< 12)': 1, 'Almuerzo (12-14)': 2,
    'Tarde (15-17)': 3, 'Noche (18-20)': 4, 'Cierre (> 20)': 5
}
df['time_block_order'] = df['time_block'].map(time_block_order_map)

print('time_block distribución:')
print(df['time_block'].value_counts())

## 4.2 — Ticket total por pedido

In [ ]:
ticket = df.groupby('order_id')['revenue'].sum().reset_index()
ticket.columns = ['order_id', 'ticket_total']
df = df.merge(ticket, on='order_id', how='left')

print(f'Ticket promedio: ${df.groupby("order_id")["ticket_total"].first().mean():.2f}')
print(f'Ticket máximo:   ${df["ticket_total"].max():.2f}')

## 4.3 — Tamaño legible (para Power BI)

In [ ]:
size_map   = {'S': 'Small', 'M': 'Medium', 'L': 'Large', 'XL': 'X-Large', 'XXL': 'XX-Large'}
size_order = {'Small': 1, 'Medium': 2, 'Large': 3, 'X-Large': 4, 'XX-Large': 5}

df['size_label'] = df['size'].map(size_map)
df['size_order'] = df['size_label'].map(size_order)

print('Shape final:', df.shape)
print('Columnas:', df.columns.tolist())

## 4.4 — Guardar

In [ ]:
df.to_csv(CLEAN + 'pizza_features.csv', index=False, encoding='utf-8')
print('pizza_features.csv guardado —', df.shape[0], 'filas x', df.shape[1], 'columnas')